# 모델 학습

auth, post, comment, frontend의 KD-CNN과 OCSVM을 학습한다.
서비스별 teacher를 공유해 학생 구조 5종(`1x8`, `2x8`, `1x16`, `2x16`, `2x32`)을 비교한다.

모델은 `models_<arch>/`에 저장하고, ZIP 파일은 Drive가 연결되어 있으면 함께 복사한다.

## 환경 준비

`training_colab.zip`에 `data/`와 아래 코드 파일을 포함한다.

- `train_kd_pipeline.py`
- `student_cnn.py`
- `data_utils.py`
- `recalibrate_ocsvm.py`
- `recal_robust.py`

In [ ]:
import os, glob, zipfile, subprocess, sys

DRIVE_DIR = "/content/drive/MyDrive"

if not os.path.exists("data"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    zs = (glob.glob(f'{DRIVE_DIR}/training_colab.zip')
          + glob.glob("training_colab.zip")
          + glob.glob("/content/training_colab.zip"))
    if zs:
        print(f"압축 해제: {os.path.basename(zs[0])}")
        zipfile.ZipFile(zs[0]).extractall('.')

assert os.path.exists("data"), "data/ 가 없습니다 — training_colab.zip 을 준비하세요"

need = ["train_kd_pipeline.py", "student_cnn.py", "data_utils.py", "recalibrate_ocsvm.py", "recal_robust.py"]
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"training_colab.zip 에 다음 파일이 없습니다: {missing}"

try:
    import sklearn, joblib  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib"])

import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"데이터·코드 확인 완료 | torch={torch.__version__} | device={dev}")
if dev == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU 런타임")

## 학습 설정

Teacher는 `deep`, 학습 상한은 120,000개, 배치 크기는 2,048, 시드는 42다.
`train_seeded.py`에서 Python, NumPy, PyTorch 시드를 설정한다.

In [ ]:
import os, sys, json, shutil, subprocess, time
import numpy as np

SERVICES = ["auth", "post", "comment", "frontend"]
ARCHS    = ["1x8", "2x8", "1x16", "2x16", "2x32"]

TEACHER = "deep"
LIMIT   = 120000
BATCH   = 2048
SEED    = 42
MASK_TRANSPORT = "0"

TEACHER_DIR = "teachers"
DRIVE_DIR   = "/content/drive/MyDrive"

def models_root(arch): return f"models_{arch}"

SHIM = '''"""시드를 설정하고 학습을 시작한다."""
import os, random, sys
import numpy as np
import torch

SEED = int(os.environ.get("TRAIN_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

import train_kd_pipeline as T
T.main()
'''
open("train_seeded.py", "w", encoding="utf-8").write(SHIM)

def _train(svc, arch, out_dir, teacher_pth=None):
    """teacher_pth가 있으면 기존 teacher로 학습한다."""
    cmd = (f"MASK_TRANSPORT={MASK_TRANSPORT} TRAIN_SEED={SEED} "
           f"python train_seeded.py --data data/{svc} --out {out_dir} "
           f"--arch {arch} --teacher {TEACHER} --limit {LIMIT} "
           f"--batch-size {BATCH} --seed {SEED}")
    if teacher_pth:
        cmd += f" --teacher-pth {teacher_pth}"
    return os.system(cmd)

def prepare_teachers(force=False):
    """서비스별 teacher를 학습해 teachers/에 저장한다."""
    os.makedirs(TEACHER_DIR, exist_ok=True)
    for svc in SERVICES:
        dst = f"{TEACHER_DIR}/{svc}.pth"
        if os.path.exists(dst) and not force:
            print(f"[teacher] {svc}: 기존 {dst} 재사용")
            continue
        tmp = f"_teacher_bootstrap/{svc}"
        print(f"\n[teacher] {svc} 학습")
        t0 = time.time()
        if _train(svc, "1x8", tmp) != 0:
            raise RuntimeError(f"{svc} teacher 학습 실패")
        shutil.copy2(f"{tmp}/teacher.pth", dst)
        print(f"[teacher] {svc} → {dst}  ({time.time()-t0:.0f}s)")
    shutil.rmtree("_teacher_bootstrap", ignore_errors=True)
    print("\n보관된 teacher:", sorted(os.listdir(TEACHER_DIR)))

def run_arch(arch, to_drive=True):
    """학생 구조 하나를 학습·재보정하고 저장한다."""
    assert arch in ARCHS, arch
    root = models_root(arch)
    t0 = time.time()

    for svc in SERVICES:
        tp = f"{TEACHER_DIR}/{svc}.pth"
        assert os.path.exists(tp), f"{tp} 없음 — prepare_teachers() 먼저 실행"
        print(f"\n[{arch}] {svc} 학습")
        if _train(svc, arch, f"{root}/{svc}", teacher_pth=tp) != 0:
            print(f"[{arch}] {svc} 학습 실패")

    print(f"\n[{arch}] 학습 산출:")
    for svc in SERVICES:
        need = ["student.pth", "student_ts.pt", "ocsvm.pkl", "threshold.json", "teacher.pth"]
        miss = [f for f in need if not os.path.exists(f"{root}/{svc}/{f}")]
        print(f"  {svc:9} {'OK' if not miss else '누락: ' + str(miss)}")

    print(f"\n[{arch}] 재보정")
    import importlib, recal_robust
    importlib.reload(recal_robust)
    results, failed = recal_robust.run(os.path.abspath(root), os.path.abspath("data"))

    zip_name = f"{root}.zip"
    if os.path.exists(zip_name):
        os.remove(zip_name)
    subprocess.run(["zip", "-r", "-q", zip_name, root])
    size = os.path.getsize(zip_name) / 1e6
    if to_drive and os.path.isdir(DRIVE_DIR):
        shutil.copy2(zip_name, f"{DRIVE_DIR}/{zip_name}")
        print(f"[저장] {zip_name}: {size:.1f}MB, Drive 복사 완료")
    else:
        print(f"[저장] {zip_name}: {size:.1f}MB")

    print(f"[{arch}] 총 {time.time()-t0:.0f}s   실패: {failed if failed else '없음'}")
    return results

print("설정 완료")
print(f"  SERVICES = {SERVICES}")
print(f"  ARCHS    = {ARCHS}")
print(f"  teacher={TEACHER}, limit={LIMIT}, batch={BATCH}, seed={SEED}")
print("  train_seeded.py 생성 완료")

## 데이터 점검

서비스별 배열 크기와 세션 수, 공격 데이터 파일을 확인한다.

In [ ]:
import os, glob, numpy as np

print("정상 데이터")
for svc in SERVICES:
    d = f"data/{svc}"
    if not os.path.exists(d):
        print(f"  {svc}/ 디렉토리 없음")
        continue
    for name in ["X_benign", "X_testbenign"]:
        p = f"{d}/{name}.npy"
        if os.path.exists(p):
            a = np.load(p, mmap_mode="r")
            sess_p = p.replace("X_", "sess_")
            n_sess = len(np.unique(np.load(sess_p))) if os.path.exists(sess_p) else "?"
            print(f"  {svc:9} {name:14} {str(a.shape):20} sessions={n_sess}")
        else:
            print(f"  {svc}/{name}.npy 없음")

print("\n공격 데이터")
for p in sorted(glob.glob("data/_attack/X_attack_*.npy")):
    svc = os.path.basename(p).replace("X_attack_", "").rsplit("_", 1)[0]
    if svc not in SERVICES:
        continue
    a = np.load(p, mmap_mode="r")
    print(f"  {os.path.basename(p):40} {a.shape}")

## 세션·특징 분포

세션 수, 분산이 작은 특징 수, 단일 세션의 비중을 확인한다.

In [ ]:
import numpy as np, os
from collections import Counter

def diagnose(svc):
    Xp = f"data/{svc}/X_benign.npy"
    Sp = f"data/{svc}/sess_benign.npy"
    if not os.path.exists(Xp):
        return f"[{svc}] SKIP (파일 없음)"
    X = np.load(Xp); S = np.load(Sp) if os.path.exists(Sp) else None
    n_img, n_feat, w = X.shape

    # 특징별 평균·분산
    feat_mean = X.mean(axis=(0, 2))
    feat_var = X.var(axis=(0, 2))
    dead_features = int((feat_var < 1e-6).sum())
    active_features = int((feat_mean > 0.01).sum())

    # 세션 다양성
    if S is not None:
        n_sess = len(np.unique(S))
        sess_sizes = Counter(S.tolist())
        max_sess = max(sess_sizes.values())
        min_sess = min(sess_sizes.values())
        med_sess = int(np.median(list(sess_sizes.values())))
    else:
        n_sess = max_sess = min_sess = med_sess = -1

    # 점검 기준
    warn = []
    if n_sess < 50: warn.append(f"세션<{50}({n_sess})")
    if dead_features > 8: warn.append(f"dead특징{dead_features}/{n_feat}")
    if max_sess > n_img * 0.3: warn.append(f"단일세션과대({max_sess}={100*max_sess/n_img:.0f}%)")

    return (f"[{svc:9}] n={n_img:>6} sess={n_sess:>4} (min/med/max={min_sess}/{med_sess}/{max_sess}) "
            f"active={active_features}/{n_feat} dead={dead_features}/{n_feat}"
            + ("  확인: " + ", ".join(warn) if warn else "  OK"))

for svc in SERVICES:
    print(diagnose(svc))

[auth     ] n=180000 sess=1326 (min/med/max=1/6/9818) active=12/20 dead=6/20  OK
[post     ] n= 16821 sess= 288 (min/med/max=1/5/196) active=14/20 dead=6/20  OK
[comment  ] n= 54027 sess= 335 (min/med/max=1/196/428) active=13/20 dead=8/20  OK
[frontend ] n=180000 sess=11707 (min/med/max=1/3/996) active=12/20 dead=6/20  OK


## Teacher 학습

서비스별 teacher를 `teachers/<svc>.pth`에 저장해 모든 학생 구조에서 재사용한다.
최초 실행은 `1x8`로 학습한 뒤 teacher만 보관한다.

In [ ]:
prepare_teachers()


[teacher] auth 학습
[teacher] auth → teachers/auth.pth  (390s)

[teacher] post 학습
[teacher] post → teachers/post.pth  (72s)

[teacher] comment 학습
[teacher] comment → teachers/comment.pth  (226s)

[teacher] frontend 학습
[teacher] frontend → teachers/frontend.pth  (456s)

보관된 teacher: ['auth.pth', 'comment.pth', 'frontend.pth', 'post.pth']


## 학생 1x8

학습·재보정 결과를 `models_1x8/`와 `models_1x8.zip`에 저장한다.

In [ ]:
RESULTS_1_8 = run_arch("1x8")

## 학생 2x8

학습·재보정 결과를 `models_2x8/`와 `models_2x8.zip`에 저장한다.

In [ ]:
RESULTS_2_8 = run_arch("2x8")

## 학생 1x16

학습·재보정 결과를 `models_1x16/`와 `models_1x16.zip`에 저장한다.

In [ ]:
RESULTS_1_16 = run_arch("1x16")

## 학생 2x16

학습·재보정 결과를 `models_2x16/`와 `models_2x16.zip`에 저장한다.

In [ ]:
RESULTS_2_16 = run_arch("2x16")

## 학생 2x32

학습·재보정 결과를 `models_2x32/`와 `models_2x32.zip`에 저장한다.

In [ ]:
RESULTS_2_32 = run_arch("2x32")

## 결과 비교

구조별 파라미터 수와 서비스별 OCSVM 설정, FPR, AUC, 탐지율을 비교한다.

In [ ]:
import os, json
import torch
from student_cnn import make_student

rows = []
for arch in ARCHS:
    root = models_root(arch)
    n_params = sum(p.numel() for p in make_student(arch).parameters())
    for svc in SERVICES:
        tp = f"{root}/{svc}/threshold.json"
        if not os.path.exists(tp):
            rows.append((arch, n_params, svc, None)); continue
        m = json.load(open(tp, encoding="utf-8"))
        rows.append((arch, n_params, svc, m))

print(f"{'arch':7s}{'params':>10s}  {'svc':9s}{'gamma':>7s}{'nu':>6s}{'thr':>11s}{'valFPR':>8s}"
      f"{'worstAUC':>10s}  detect recall")
print("-" * 118)
for arch, n_params, svc, m in rows:
    if m is None:
        print(f"{arch:7s}{n_params:10,d}  {svc:9s}  (없음)"); continue
    dr = m.get("detect_recall") or {}
    dr_s = " ".join(f"{k}={v*100:.0f}%" for k, v in list(dr.items())[:4])
    print(f"{arch:7s}{n_params:10,d}  {svc:9s}{str(m.get('gamma')):>7s}{m.get('nu', 0):>6.2f}"
          f"{m.get('threshold_df', float('nan')):11.3f}{m.get('val_fpr_at_thr', 0):>8.3f}"
          f"{m.get('worst_detect_auc', float('nan')):10.4f}  {dr_s}")

print("\n생성된 zip:")
for arch in ARCHS:
    z = f"{models_root(arch)}.zip"
    if os.path.exists(z):
        print(f"  {z:22s} {os.path.getsize(z)/1e6:6.1f}MB")